In [1]:
from pathlib import Path
from collections import Counter

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
DRIVE = BASE / "inputs" / "drive"

archivos = [
    p for p in DRIVE.rglob("*")
    if p.is_file()
]

print("=== INVENTARIO ORIGINAL ===")
print("Archivos encontrados:", len(archivos))

conteo = Counter(
    p.suffix.lower() for p in archivos
)

print("\n=== FORMATOS ===")
for extension, cantidad in sorted(conteo.items()):
    print(extension or "(sin extensión)", cantidad)

print("\n=== TAMAÑO TOTAL ===")
print(
    "Bytes:",
    sum(p.stat().st_size for p in archivos)
)

print("\n=== ARCHIVOS IDENTIFICADOS ===")
for archivo in sorted(archivos):
    print(
        archivo.suffix.lower(),
        archivo.stat().st_size,
        archivo.relative_to(DRIVE)
    )

=== INVENTARIO ORIGINAL ===
Archivos encontrados: 65

=== FORMATOS ===
.csv 8
.doc 2
.docx 2
.json 5
.pdf 44
.txt 1
.xls 3

=== TAMAÑO TOTAL ===
Bytes: 135127109

=== ARCHIVOS IDENTIFICADOS ===
.xls 806912 Datos Estructurados/Afiliación y Traspasos (Desde 1993).XLS
.xls 1161216 Datos Estructurados/Inversiones y Rentabilidad (Desde 1993).XLS
.csv 38437 Datos Estructurados/Las 12 series PN01173MM–182MM, 184MM y 198MM.csv
.csv 130000 Datos Estructurados/Las 26 restantes PN01167MM–172MM, 183MM, 185MM–197MM, 199MM–204MM.csv
.csv 7739 Datos Estructurados/Tipo de cambio - promedio del periodo.csv
.xls 1153161 Datos Estructurados/Valores cuota (Desde 1993).XLS
.csv 9736 Datos Estructurados/Índice de precios Lima Metropolitana.csv
.pdf 4231671 Datos No estructurados/Memoria Habitat/MEOE41322015AIA01.PDF
.pdf 2104516 Datos No estructurados/Memoria Habitat/MEOE41322016AIA01.PDF
.pdf 1834463 Datos No estructurados/Memoria Habitat/MEOE41322017AIA01.PDF
.pdf 2968656 Datos No estructurados/Memoria Ha

In [2]:
import csv

print("=== CONTEO DE REGISTROS CSV ===")

total_filas_csv = 0

for archivo in sorted(DRIVE.rglob("*")):

    if not archivo.is_file():
        continue

    if archivo.suffix.lower() != ".csv":
        continue

    try:
        with archivo.open(
            "r",
            encoding="latin-1",
            newline=""
        ) as f:
            cantidad = sum(1 for _ in csv.reader(f))

        total_filas_csv += cantidad

        print(
            archivo.name,
            "->",
            cantidad,
            "filas físicas CSV"
        )

    except Exception as e:
        print("ERROR:", archivo.name, e)

print("\nTOTAL DE FILAS CSV:", total_filas_csv)

=== CONTEO DE REGISTROS CSV ===
Las 12 series PN01173MM–182MM, 184MM y 198MM.csv -> 403 filas físicas CSV
Las 26 restantes PN01167MM–172MM, 183MM, 185MM–197MM, 199MM–204MM.csv -> 401 filas físicas CSV
Tipo de cambio - promedio del periodo.csv -> 384 filas físicas CSV
Índice de precios Lima Metropolitana.csv -> 403 filas físicas CSV
bcrp_json_rentabilidad_real_12m_aplanada.csv -> 141 filas físicas CSV
manifiesto_memorias.csv -> 49 filas físicas CSV
revision_texto.csv -> 49 filas físicas CSV
tabla_piloto_terminos.csv -> 45 filas físicas CSV

TOTAL DE FILAS CSV: 1875


In [3]:
import pandas as pd
from pathlib import Path

exceles = sorted(
    p for p in DRIVE.rglob("*")
    if p.is_file() and p.suffix.lower() in (".xls", ".xlsx")
)

print("=== AUDITORÍA DE HOJAS EXCEL SBS ===")

resultados_excel = []
total_filas_excel = 0

for ruta in exceles:
    print(f"\nARCHIVO: {ruta.name}")

    libro = None
    ultimo_error = None

    for motor in ("xlrd", "openpyxl"):
        try:
            libro = pd.ExcelFile(ruta, engine=motor)
            print("Motor de lectura:", motor)
            break
        except Exception as e:
            ultimo_error = str(e)

    if libro is None:
        print("ERROR AL ABRIR:", ultimo_error)
        continue

    for hoja in libro.sheet_names:
        datos = pd.read_excel(
            libro,
            sheet_name=hoja,
            header=None,
            dtype=object
        )

        filas_fisicas = len(datos)
        filas_con_contenido = int(
            datos.notna().any(axis=1).sum()
        )
        celdas_con_contenido = int(
            datos.notna().sum().sum()
        )

        resultados_excel.append({
            "archivo": ruta.name,
            "hoja": hoja,
            "filas_fisicas": filas_fisicas,
            "filas_con_contenido": filas_con_contenido,
            "columnas": len(datos.columns),
            "celdas_con_contenido": celdas_con_contenido
        })

        total_filas_excel += filas_con_contenido

        print(
            f"  Hoja: {hoja} | "
            f"Filas con contenido: {filas_con_contenido} | "
            f"Columnas: {len(datos.columns)} | "
            f"Celdas con contenido: {celdas_con_contenido}"
        )

resumen_excel = pd.DataFrame(resultados_excel)

print("\n=== RESUMEN FINAL EXCEL ===")
print(resumen_excel.to_string(index=False))

print("\nTOTAL DE HOJAS:", len(resumen_excel))
print("TOTAL FILAS CON CONTENIDO:", total_filas_excel)
print(
    "TOTAL CELDAS CON CONTENIDO:",
    int(resumen_excel["celdas_con_contenido"].sum())
)

=== AUDITORÍA DE HOJAS EXCEL SBS ===

ARCHIVO: Afiliación y Traspasos (Desde 1993).XLS
ERROR AL ABRIR: File is not a zip file

ARCHIVO: Inversiones y Rentabilidad (Desde 1993).XLS
ERROR AL ABRIR: File is not a zip file

ARCHIVO: Valores cuota (Desde 1993).XLS
Motor de lectura: openpyxl
  Hoja: Valor cuota diario | Filas con contenido: 8542 | Columnas: 17 | Celdas con contenido: 84620

=== RESUMEN FINAL EXCEL ===
                       archivo               hoja  filas_fisicas  filas_con_contenido  columnas  celdas_con_contenido
Valores cuota (Desde 1993).XLS Valor cuota diario           8543                 8542        17                 84620

TOTAL DE HOJAS: 1
TOTAL FILAS CON CONTENIDO: 8542
TOTAL CELDAS CON CONTENIDO: 84620


In [4]:
import json

print("=== AUDITORÍA DE JSON BCRP ===")

total_observaciones_json = 0

for ruta in sorted(DRIVE.rglob("*.json")):

    with ruta.open("r", encoding="utf-8") as archivo:
        contenido = json.load(archivo)

    registros = None

    if isinstance(contenido, dict):
        for clave in ("periods", "data", "observations"):
            valor = contenido.get(clave)
            if isinstance(valor, list):
                registros = valor
                break

        if registros is None:
            for valor in contenido.values():
                if isinstance(valor, dict):
                    candidato = valor.get("periods")
                    if isinstance(candidato, list):
                        registros = candidato
                        break

    elif isinstance(contenido, list):
        registros = contenido

    if registros is None:
        print(ruta.name, "-> revisar estructura:", type(contenido).__name__)
        continue

    cantidad = len(registros)
    total_observaciones_json += cantidad

    print(ruta.name, "->", cantidad, "observaciones")

print("\nTOTAL OBSERVACIONES JSON:", total_observaciones_json)

=== AUDITORÍA DE JSON BCRP ===
PN01178MM_Sistema.json -> 139 observaciones
PN01179MM_Habitat.json -> 140 observaciones
PN01180MM_Integra.json -> 140 observaciones
PN01181MM_Prima.json -> 140 observaciones
PN01182MM_Profuturo.json -> 140 observaciones

TOTAL OBSERVACIONES JSON: 699


In [5]:
import importlib.util

print("xlrd:", importlib.util.find_spec("xlrd"))
print("openpyxl:", importlib.util.find_spec("openpyxl"))

xlrd: ModuleSpec(name='xlrd', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7fcc18763010>, origin='/home/junior/BigData_UPAO/bigdata_env/lib/python3.11/site-packages/xlrd/__init__.py', submodule_search_locations=['/home/junior/BigData_UPAO/bigdata_env/lib/python3.11/site-packages/xlrd'])
openpyxl: ModuleSpec(name='openpyxl', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7fcc199b3b10>, origin='/home/junior/BigData_UPAO/bigdata_env/lib/python3.11/site-packages/openpyxl/__init__.py', submodule_search_locations=['/home/junior/BigData_UPAO/bigdata_env/lib/python3.11/site-packages/openpyxl'])


In [6]:
from pathlib import Path
import pandas as pd
import zipfile

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
DRIVE = BASE / "inputs" / "drive"

archivos_excel = sorted(
    p for p in DRIVE.rglob("*")
    if p.is_file()
    and p.suffix.lower() in (".xls", ".xlsx")
)

resultados_excel = []
errores_excel = []

print("=== AUDITORÍA COMPLETA DE EXCEL SBS ===")

for ruta in archivos_excel:

    print("\nARCHIVO:", ruta.name)

    # Los XLSX son ZIP internamente.
    # Los XLS antiguos usan otro formato.
    motor = "openpyxl" if zipfile.is_zipfile(ruta) else "xlrd"

    print("Motor detectado:", motor)

    try:
        libro = pd.ExcelFile(ruta, engine=motor)

        print("Hojas encontradas:", len(libro.sheet_names))

        for hoja in libro.sheet_names:

            datos = pd.read_excel(
                libro,
                sheet_name=hoja,
                header=None,
                dtype=object
            )

            filas = len(datos)

            filas_con_contenido = int(
                datos.notna().any(axis=1).sum()
            )

            celdas_con_contenido = int(
                datos.notna().sum().sum()
            )

            resultados_excel.append({
                "archivo": ruta.name,
                "hoja": hoja,
                "filas_fisicas": filas,
                "filas_con_contenido": filas_con_contenido,
                "columnas": len(datos.columns),
                "celdas_con_contenido": celdas_con_contenido
            })

            print(
                f"  {hoja}: "
                f"{filas_con_contenido} filas con contenido, "
                f"{len(datos.columns)} columnas, "
                f"{celdas_con_contenido} celdas ocupadas"
            )

    except Exception as e:
        errores_excel.append({
            "archivo": ruta.name,
            "error": str(e)
        })
        print("ERROR:", repr(e))

resumen_excel = pd.DataFrame(resultados_excel)

print("\n========== RESUMEN ==========")

if not resumen_excel.empty:

    print(resumen_excel.to_string(index=False))

    print("\nTOTAL DE ARCHIVOS EXCEL:", len(archivos_excel))

    print("TOTAL DE HOJAS:", len(resumen_excel))

    print(
        "TOTAL DE FILAS CON CONTENIDO:",
        int(resumen_excel["filas_con_contenido"].sum())
    )

    print(
        "TOTAL DE CELDAS CON CONTENIDO:",
        int(resumen_excel["celdas_con_contenido"].sum())
    )

print("\nARCHIVOS CON ERROR:", len(errores_excel))

for error in errores_excel:
    print(error)

=== AUDITORÍA COMPLETA DE EXCEL SBS ===

ARCHIVO: Afiliación y Traspasos (Desde 1993).XLS
Motor detectado: xlrd
Hojas encontradas: 8
  Afil Activos: 380 filas con contenido, 12 columnas, 2495 celdas ocupadas
  Afiliacion mensual: 404 filas con contenido, 12 columnas, 2715 celdas ocupadas
  Mov Fondo x Traspasos-Ingresos : 346 filas con contenido, 10 columnas, 2304 celdas ocupadas
  Mov Fondo x Traspasos-Salidas: 346 filas con contenido, 9 columnas, 2466 celdas ocupadas
  Mov Fondo x Traspasos-Neto: 346 filas con contenido, 9 columnas, 1963 celdas ocupadas
  Traspasos Afiliados-Ingresos: 345 filas con contenido, 9 columnas, 2303 celdas ocupadas
  Traspasos Afiliados-Salidas: 345 filas con contenido, 9 columnas, 2303 celdas ocupadas
  Traspasos Afiliados-Neto: 345 filas con contenido, 8 columnas, 1963 celdas ocupadas

ARCHIVO: Inversiones y Rentabilidad (Desde 1993).XLS
Motor detectado: xlrd
Hojas encontradas: 6
  Rentabilidad Nominal F2: 345 filas con contenido, 139 columnas, 2205 celda

In [7]:
import pandas as pd
import zipfile

print("=== AUDITORÍA DE VALORES NUMÉRICOS SBS ===")

resultados_numericos = []

for ruta in archivos_excel:

    motor = "openpyxl" if zipfile.is_zipfile(ruta) else "xlrd"

    libro = pd.ExcelFile(ruta, engine=motor)

    for hoja in libro.sheet_names:

        datos = pd.read_excel(
            libro,
            sheet_name=hoja,
            header=None,
            dtype=object
        )

        # Contar valores que ya son numéricos.
        # Se excluyen booleanos y celdas vacías.
        import numbers

        cantidad_numerica = sum(
            isinstance(valor, numbers.Number)
            and not isinstance(valor, bool)
            and pd.notna(valor)
            for valor in datos.to_numpy().flat
        )

        resultados_numericos.append({
            "archivo": ruta.name,
            "hoja": hoja,
            "valores_numericos": cantidad_numerica
        })

resumen_numericos = pd.DataFrame(resultados_numericos)

print(resumen_numericos.to_string(index=False))

print(
    "\nTOTAL DE VALORES NUMÉRICOS SBS:",
    int(resumen_numericos["valores_numericos"].sum())
)

=== AUDITORÍA DE VALORES NUMÉRICOS SBS ===
                                    archivo                            hoja  valores_numericos
    Afiliación y Traspasos (Desde 1993).XLS                    Afil Activos               2105
    Afiliación y Traspasos (Desde 1993).XLS              Afiliacion mensual               2300
    Afiliación y Traspasos (Desde 1993).XLS Mov Fondo x Traspasos-Ingresos                1950
    Afiliación y Traspasos (Desde 1993).XLS   Mov Fondo x Traspasos-Salidas               2112
    Afiliación y Traspasos (Desde 1993).XLS      Mov Fondo x Traspasos-Neto               1611
    Afiliación y Traspasos (Desde 1993).XLS    Traspasos Afiliados-Ingresos               1950
    Afiliación y Traspasos (Desde 1993).XLS     Traspasos Afiliados-Salidas               1950
    Afiliación y Traspasos (Desde 1993).XLS        Traspasos Afiliados-Neto               1611
Inversiones y Rentabilidad (Desde 1993).XLS         Rentabilidad Nominal F2               1692
Inversi

In [8]:
print("=== VALORES NUMERICOS POR HOJA ===")

resumen_numericos.sort_values(
    "valores_numericos",
    ascending=False
).to_string(index=False)

=== VALORES NUMERICOS POR HOJA ===


'                                    archivo                            hoja  valores_numericos\n             Valores cuota (Desde 1993).XLS              Valor cuota diario              76059\nInversiones y Rentabilidad (Desde 1993).XLS              Cartera por moneda               4849\nInversiones y Rentabilidad (Desde 1993).XLS Rentabilidad Nominal F0 F1 y F3               2801\nInversiones y Rentabilidad (Desde 1993).XLS    Rentabilidad Real F0 F1 y F3               2801\n    Afiliación y Traspasos (Desde 1993).XLS              Afiliacion mensual               2300\n    Afiliación y Traspasos (Desde 1993).XLS   Mov Fondo x Traspasos-Salidas               2112\n    Afiliación y Traspasos (Desde 1993).XLS                    Afil Activos               2105\nInversiones y Rentabilidad (Desde 1993).XLS            Rentabilidad Real F2               1985\n    Afiliación y Traspasos (Desde 1993).XLS Mov Fondo x Traspasos-Ingresos                1950\n    Afiliación y Traspasos (Desde 1993)

In [10]:
import pandas as pd
from pathlib import Path

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
DRIVE = BASE / "inputs" / "drive"

archivo = next(
    p for p in DRIVE.rglob("*")
    if p.is_file()
    and "Afiliación y Traspasos" in p.name
)

libro = pd.ExcelFile(archivo, engine="xlrd")

print("=== HOJAS REALES DEL ARCHIVO ===")

for nombre in libro.sheet_names:
    print(repr(nombre))

hojas_deseadas = [
    "Mov Fondo x Traspasos-Ingresos",
    "Traspasos Afiliados-Ingresos"
]

for nombre_buscado in hojas_deseadas:

    coincidencias = [
        nombre_real
        for nombre_real in libro.sheet_names
        if nombre_real.strip() == nombre_buscado.strip()
    ]

    if not coincidencias:
        print("NO ENCONTRADA:", nombre_buscado)
        continue

    nombre_real = coincidencias[0]

    datos = pd.read_excel(
        libro,
        sheet_name=nombre_real,
        header=None,
        dtype=object
    )

    print("\n" + "=" * 60)
    print("HOJA:", repr(nombre_real))
    print("FILAS:", len(datos))
    print("COLUMNAS:", len(datos.columns))
    print("=" * 60)

    print(
        datos.head(15).to_string(
            index=False,
            header=False
        )
    )

=== HOJAS REALES DEL ARCHIVO ===
'Afil Activos'
'Afiliacion mensual'
'Mov Fondo x Traspasos-Ingresos '
'Mov Fondo x Traspasos-Salidas'
'Mov Fondo x Traspasos-Neto'
'Traspasos Afiliados-Ingresos'
'Traspasos Afiliados-Salidas'
'Traspasos Afiliados-Neto'

HOJA: 'Mov Fondo x Traspasos-Ingresos '
FILAS: 347
COLUMNAS: 10
NaN MOVIMIENTO DEL FONDOS DE PENSIONES POR TRASPASOS EFECTIVOS - INGRESOS     NaN       NaN       NaN     NaN       NaN              NaN       NaN        NaN
NaN                                            (En miles de nuevos soles)     NaN       NaN       NaN     NaN       NaN              NaN       NaN        NaN
NaN                                                              INGRESOS     NaN       NaN       NaN     NaN       NaN              NaN       NaN        NaN
NaN                                                                 Fecha Habitat Horizonte   Integra Prima1/ Profuturo Unión Vida 1/ 2/     Unión        SPP
NaN                                                

In [11]:
import pandas as pd
import zipfile

print("=== INSPECCIÓN DE LAS 15 HOJAS SBS ===")

for ruta in archivos_excel:

    motor = "openpyxl" if zipfile.is_zipfile(ruta) else "xlrd"

    libro = pd.ExcelFile(ruta, engine=motor)

    print("\n" + "=" * 75)
    print("ARCHIVO:", ruta.name)
    print("=" * 75)

    for hoja in libro.sheet_names:

        df = pd.read_excel(
            libro,
            sheet_name=hoja,
            header=None,
            dtype=object
        )

        print("\nHOJA:", repr(hoja))
        print("DIMENSIONES:", df.shape)

        # Mostrar encabezados y primeras observaciones
        print(
            df.head(8).to_string(
                index=True,
                header=True,
                max_cols=12
            )
        )

=== INSPECCIÓN DE LAS 15 HOJAS SBS ===

ARCHIVO: Afiliación y Traspasos (Desde 1993).XLS

HOJA: 'Afil Activos'
DIMENSIONES: (382, 12)
                            0        1           2        3         4          5              6         7            8             9               10         11
0  NÚMERO DE AFILIADOS ACTIVOS      NaN         NaN      NaN       NaN        NaN            NaN       NaN          NaN           NaN             NaN        NaN
1                          NaN      NaN         NaN      NaN       NaN        NaN            NaN       NaN          NaN           NaN             NaN        NaN
2                          NaN  Habitat  Horizonte   Integra  Prima 1/  Profuturo  Unión Vida 2/  Unión 3/  El Roble 4/  Megafondo 6/  Providencia 5/  Total SPP
3          1993-12-01 00:00:00      NaN       76552   137353       NaN     163448          19728     75944        77606          5163           70607     626401
4          1994-12-01 00:00:00      NaN      222038   208215 